# Assignment 1 Results Comparison

This notebook loads the saved JSON logs for Assignment 1 and compares results across:

- FAISS + MiniLM + Llama
- FAISS + BGE + Llama
- Chroma + MiniLM + Llama
- Chroma + BGE + Llama

It also reads the `compare-models` logs for the two-model generator comparison.

In [8]:
from pathlib import Path
import json
import re

import pandas as pd
from IPython.display import display

pd.set_option("display.max_colwidth", 200)

ROOT = Path('.')
ASK_DIRS = [
    ROOT / 'FAISS_Outputs',
    ROOT / 'FAISS_Outputs' / 'BGE',
    ROOT / 'Chromo_Output',
    ROOT / 'Chromo_Output' / 'MiniLM',
]
COMPARE_DIR = ROOT / 'Compare'


In [9]:
def short_embedding(model_name: str) -> str:
    model_name = (model_name or '').lower()
    if 'bge' in model_name:
        return 'BGE'
    if 'minilm' in model_name:
        return 'MiniLM'
    return model_name.split('/')[-1] or 'unknown'


def normalize_question(question: str) -> str:
    q = re.sub(r'\s+', ' ', (question or '').strip().lower())
    if 'iot networks' in q and 'evaluation' in q:
        return 'iot_evaluation'
    if 'iot networks' in q and ('topics' in q or 'covered' in q):
        return 'iot_topics'
    if 'applied machine learning in health care' in q and 'evaluation' in q:
        return 'healthcare_evaluation'
    if 'applied machine learning in health care' in q and 'credits' in q:
        return 'healthcare_credits'
    if 'theory of computation' in q and 'credits' in q:
        return 'toc_credits'
    if 'theory of computation' in q and 'evaluation' in q:
        return 'toc_evaluation'
    if 'generative ai' in q and 'evaluation' in q:
        return 'genai_evaluation'
    return q


QUESTION_LABELS = {
    'iot_evaluation': 'IoT evaluation criteria',
    'iot_topics': 'IoT topics covered',
    'healthcare_evaluation': 'Healthcare evaluation criteria',
    'healthcare_credits': 'Healthcare credits',
    'toc_credits': 'Theory of Computation credits',
    'toc_evaluation': 'Theory of Computation evaluation criteria',
    'genai_evaluation': 'Generative AI evaluation criteria',
}


FINAL_MATRIX_ORDER = [
    'iot_evaluation',
    'iot_topics',
    'healthcare_evaluation',
    'healthcare_credits',
]


def config_label(vector_store: str, embedding_model: str) -> str:
    return f"{vector_store.upper()} + {short_embedding(embedding_model)} + Llama"


def load_ask_runs(paths: list[Path]) -> pd.DataFrame:
    rows = []
    for base in paths:
        if not base.exists():
            continue
        for path in sorted(base.rglob('*.json')):
            payload = json.loads(path.read_text(encoding='utf-8'))
            if 'answer' not in payload:
                continue
            retrieved_chunks = payload.get('retrieved_chunks', [])
            top_source = retrieved_chunks[0].get('source') if retrieved_chunks else None
            rows.append({
                'file': path.name,
                'folder': str(path.parent.relative_to(ROOT)),
                'question': payload.get('question'),
                'question_key': normalize_question(payload.get('question', '')),
                'question_label': QUESTION_LABELS.get(normalize_question(payload.get('question', '')), payload.get('question')),
                'vector_store': payload.get('vector_store'),
                'embedding_model': payload.get('embedding_model'),
                'embedding_short': short_embedding(payload.get('embedding_model', '')),
                'config': config_label(payload.get('vector_store', 'unknown'), payload.get('embedding_model', 'unknown')),
                'generator_model': payload.get('generator_model'),
                'answer': payload.get('answer'),
                'top_source': top_source,
            })
    return pd.DataFrame(rows)


def load_compare_runs(compare_dir: Path) -> pd.DataFrame:
    rows = []
    if not compare_dir.exists():
        return pd.DataFrame(rows)
    for path in sorted(compare_dir.rglob('*.json')):
        payload = json.loads(path.read_text(encoding='utf-8'))
        for model_output in payload.get('model_outputs', []):
            rows.append({
                'file': path.name,
                'question': payload.get('question'),
                'question_key': normalize_question(payload.get('question', '')),
                'question_label': QUESTION_LABELS.get(normalize_question(payload.get('question', '')), payload.get('question')),
                'generator_backend': payload.get('generator_backend'),
                'model': model_output.get('model'),
                'answer': model_output.get('answer'),
                'sources': ', '.join(model_output.get('sources', [])),
            })
    return pd.DataFrame(rows)


In [10]:
ask_df = load_ask_runs(ASK_DIRS)
compare_df = load_compare_runs(COMPARE_DIR)

print('Ask logs:', len(ask_df))
print('Compare-model logs:', len(compare_df))

display(ask_df[['question_label', 'config', 'top_source', 'file']].sort_values(['question_label', 'config']).reset_index(drop=True))


Ask logs: 28
Compare-model logs: 14


,question_label,config,top_source,file
0,Generative AI evaluation criteria,FAISS + BGE + Llama,Generative AI and LLMs_CourseHandout.pdf,ask_20260403_164659_what_is_the_evaluation_criteria_for_generative_ai_.json
1,Generative AI evaluation criteria,FAISS + BGE + Llama,Generative AI and LLMs_CourseHandout.pdf,ask_20260403_164659_what_is_the_evaluation_criteria_for_generative_ai_.json
2,Generative AI evaluation criteria,FAISS + MiniLM + Llama,Generative AI and LLMs_CourseHandout.pdf,ask_20260403_163014_what_is_the_evaluation_criteria_for_generative_ai_.json
3,Healthcare credits,CHROMA + BGE + Llama,2025_course_handout_AMCH.pdf,ask_20260403_203009_chroma_baai_bge_base_en_v1_5_how_many_credits_are_there_in_applied_machine_lear.json
4,Healthcare credits,CHROMA + MiniLM + Llama,2025_course_handout_AMCH.pdf,ask_20260404_133634_chroma_sentence_transformers_all_minilm_l6_v2_how_many_credits_are_there_in_applied_machine_lear.json
5,Healthcare credits,CHROMA + MiniLM + Llama,2025_course_handout_AMCH.pdf,ask_20260404_133634_chroma_sentence_transformers_all_minilm_l6_v2_how_many_credits_are_there_in_applied_machine_lear.json
6,Healthcare credits,FAISS + BGE + Llama,2025_course_handout_AMCH.pdf,ask_20260404_133532_faiss_baai_bge_base_en_v1_5_how_many_credits_are_there_in_applied_machine_lear.json
7,Healthcare credits,FAISS + BGE + Llama,2025_course_handout_AMCH.pdf,ask_20260404_133532_faiss_baai_bge_base_en_v1_5_how_many_credits_are_there_in_applied_machine_lear.json
8,Healthcare credits,FAISS + MiniLM + Llama,2025_course_handout_AMCH.pdf,ask_20260404_132535_faiss_sentence_transformers_all_minilm_l6_v2_how_many_credits_are_there_in_applied_machine_lear.json
9,Healthcare evaluation criteria,CHROMA + BGE + Llama,2025_course_handout_AMCH.pdf,ask_20260403_202848_chroma_baai_bge_base_en_v1_5_what_is_the_evaluation_criteria_for_applied_machin.json


In [11]:
matrix_df = ask_df[ask_df['question_key'].isin(FINAL_MATRIX_ORDER)].copy()
matrix_df['question_label'] = pd.Categorical(
    matrix_df['question_label'],
    categories=[QUESTION_LABELS[key] for key in FINAL_MATRIX_ORDER],
    ordered=True,
)
matrix_df = matrix_df.sort_values(['question_label', 'config'])

answer_matrix = matrix_df.pivot_table(
    index='question_label',
    columns='config',
    values='answer',
    aggfunc='first',
)

source_matrix = matrix_df.pivot_table(
    index='question_label',
    columns='config',
    values='top_source',
    aggfunc='first',
)

display(answer_matrix)
display(source_matrix)


config,CHROMA + BGE + Llama,CHROMA + MiniLM + Llama,FAISS + BGE + Llama,FAISS + MiniLM + Llama
question_label,,,,
IoT evaluation criteria,"The evaluation criteria for IoT Networks, Architectures and Applications involves a final grade based on the marks/grades obtained in the mid-term and end-term evaluations along with other compone...",NaN,"The evaluation criteria for IoT Networks, Architectures and Applications is based on the marks/grades obtained in the mid-term and end-term evaluations along with other components as defined in th...","The evaluation criteria for IoT Networks, Architectures and Applications involves a combination of mid-term and end-term evaluations, along with other components, with a minimum of 40% marks requi..."
IoT topics covered,"The topics covered in IoT Networks, Architectures and Applications include basics of networking, OSI and TCP/IP models, addressing, and TCP connections; what is IoT and why is IoT; applications of...",NaN,"The topics covered in IoT Networks, Architectures and Applications include basics of networking, OSI and TCP/IP models, addressing, and TCP connections; what is IoT and why is IoT; applications of...","The topics covered in IoT Networks, Architectures and Applications include basics of networking, OSI and TCP/IP models, addressing, and TCP connections; what is IoT and why is IoT; applications of..."
Healthcare evaluation criteria,"The evaluation criteria for Applied Machine Learning in Health Care involves a relative grading method, where the final grade will be based on the marks obtained in various assessment components, ...",NaN,"The evaluation criteria for Applied Machine Learning in Health Care involves a relative grading method, where the final grade will be based on the marks obtained in various assessment components, ...","The evaluation criteria for Applied Machine Learning in Health Care involves a relative grading method, where the final grade will be based on the marks obtained in various assessment components, ..."
Healthcare credits,"There are 3 credits in Applied Machine Learning in Health Care.\nThis information is supported by multiple contexts, including Context 1 and Context 2, which both state that the course ""Applied Ma...","There are 3 credits in Applied Machine Learning in Health Care.\nThis information is supported by multiple contexts, including the course title and credits listed in Context 1 and Context 2, and t...","There are 3 credits in Applied Machine Learning in Health Care.\nThis information is supported by multiple contexts, including the course title and credits listed in Context 1 and Context 2, which...","There are 3 credits in Applied Machine Learning in Health Care.\nThis information is supported by multiple contexts, including the course title and credits listed in Context 1 and Context 2, and t..."


config,CHROMA + BGE + Llama,CHROMA + MiniLM + Llama,FAISS + BGE + Llama,FAISS + MiniLM + Llama
question_label,,,,
IoT evaluation criteria,"IoT Networks, Architectures and Applications-CSE 2023 Batch.pdf",NaN,"IoT Networks, Architectures and Applications-CSE 2023 Batch.pdf","IoT Networks, Architectures and Applications-CSE 2023 Batch.pdf"
IoT topics covered,"IoT Networks, Architectures and Applications-CSE 2023 Batch.pdf",NaN,"IoT Networks, Architectures and Applications-CSE 2023 Batch.pdf","IoT Networks, Architectures and Applications-CSE 2023 Batch.pdf"
Healthcare evaluation criteria,2025_course_handout_AMCH.pdf,NaN,2025_course_handout_AMCH.pdf,2025_course_handout_AMCH.pdf
Healthcare credits,2025_course_handout_AMCH.pdf,2025_course_handout_AMCH.pdf,2025_course_handout_AMCH.pdf,2025_course_handout_AMCH.pdf


In [12]:
detail_columns = [
    'question_label',
    'config',
    'embedding_model',
    'generator_model',
    'top_source',
    'answer',
]
display(matrix_df[detail_columns].reset_index(drop=True))


,question_label,config,embedding_model,generator_model,top_source,answer
0,IoT evaluation criteria,CHROMA + BGE + Llama,BAAI/bge-base-en-v1.5,meta-llama/Llama-3.1-8B-Instruct,"IoT Networks, Architectures and Applications-CSE 2023 Batch.pdf","The evaluation criteria for IoT Networks, Architectures and Applications involves a final grade based on the marks/grades obtained in the mid-term and end-term evaluations along with other compone..."
1,IoT evaluation criteria,FAISS + BGE + Llama,BAAI/bge-base-en-v1.5,meta-llama/Llama-3.1-8B-Instruct,"IoT Networks, Architectures and Applications-CSE 2023 Batch.pdf","The evaluation criteria for IoT Networks, Architectures and Applications is based on the marks/grades obtained in the mid-term and end-term evaluations along with other components as defined in th..."
2,IoT evaluation criteria,FAISS + BGE + Llama,BAAI/bge-base-en-v1.5,meta-llama/Llama-3.1-8B-Instruct,"IoT Networks, Architectures and Applications-CSE 2023 Batch.pdf","The evaluation criteria for IoT Networks, Architectures and Applications is based on the marks/grades obtained in the mid-term and end-term evaluations along with other components as defined in th..."
3,IoT evaluation criteria,FAISS + MiniLM + Llama,sentence-transformers/all-MiniLM-L6-v2,meta-llama/Llama-3.1-8B-Instruct,"IoT Networks, Architectures and Applications-CSE 2023 Batch.pdf","The evaluation criteria for IoT Networks, Architectures and Applications involves a combination of mid-term and end-term evaluations, along with other components, with a minimum of 40% marks requi..."
4,IoT topics covered,CHROMA + BGE + Llama,BAAI/bge-base-en-v1.5,meta-llama/Llama-3.1-8B-Instruct,"IoT Networks, Architectures and Applications-CSE 2023 Batch.pdf","The topics covered in IoT Networks, Architectures and Applications include basics of networking, OSI and TCP/IP models, addressing, and TCP connections; what is IoT and why is IoT; applications of..."
5,IoT topics covered,FAISS + BGE + Llama,BAAI/bge-base-en-v1.5,meta-llama/Llama-3.1-8B-Instruct,"IoT Networks, Architectures and Applications-CSE 2023 Batch.pdf","The topics covered in IoT Networks, Architectures and Applications include basics of networking, OSI and TCP/IP models, addressing, and TCP connections; what is IoT and why is IoT; applications of..."
6,IoT topics covered,FAISS + BGE + Llama,BAAI/bge-base-en-v1.5,meta-llama/Llama-3.1-8B-Instruct,"IoT Networks, Architectures and Applications-CSE 2023 Batch.pdf","The topics covered in IoT Networks, Architectures and Applications include basics of networking, OSI and TCP/IP models, addressing, and TCP connections; what is IoT and why is IoT; applications of..."
7,IoT topics covered,FAISS + MiniLM + Llama,sentence-transformers/all-MiniLM-L6-v2,meta-llama/Llama-3.1-8B-Instruct,"IoT Networks, Architectures and Applications-CSE 2023 Batch.pdf","The topics covered in IoT Networks, Architectures and Applications include basics of networking, OSI and TCP/IP models, addressing, and TCP connections; what is IoT and why is IoT; applications of..."
8,Healthcare evaluation criteria,CHROMA + BGE + Llama,BAAI/bge-base-en-v1.5,meta-llama/Llama-3.1-8B-Instruct,2025_course_handout_AMCH.pdf,"The evaluation criteria for Applied Machine Learning in Health Care involves a relative grading method, where the final grade will be based on the marks obtained in various assessment components, ..."
9,Healthcare evaluation criteria,FAISS + BGE + Llama,BAAI/bge-base-en-v1.5,meta-llama/Llama-3.1-8B-Instruct,2025_course_handout_AMCH.pdf,"The evaluation criteria for Applied Machine Learning in Health Care involves a relative grading method, where the final grade will be based on the marks obtained in various assessment components, ..."


In [13]:
if compare_df.empty:
    print('No compare-model logs found.')
else:
    compare_pivot = compare_df.pivot_table(
        index='question_label',
        columns='model',
        values='answer',
        aggfunc='first',
    )
    display(compare_pivot)
    display(compare_df[['question_label', 'model', 'sources', 'answer']].sort_values(['question_label', 'model']).reset_index(drop=True))


model,meta-llama/Llama-3.1-8B-Instruct,openai/gpt-oss-120b
question_label,,
Generative AI evaluation criteria,**Evaluation Criteria for Generative AI and LLMs:**\n\nThe evaluation criteria for Generative AI and LLMs is based on the relative grading method outlined in the university's academic regulations....,"**Evaluation criteria**\n\n- **Quiz** – 20 % of the final grade (20 MCQs, 1 mark each, 20‑30 min). \n- **Assignments 1 & 2** – each 20 % (total 40 %); programming assignments evaluated offline wi..."
Healthcare evaluation criteria,"The evaluation criteria for Applied Machine Learning in Health Care involves a relative grading method, where the final grade will be based on the marks obtained in various assessment components, ...","The course is assessed through four components – two quizzes worth a combined 20 % (each quiz 10 %), an assignment worth 20 %, a mid‑term presentation worth 20 % and a final project worth 40 %, wi..."
How many quiz are there in machine learning for healthcare?,There is one quiz in the Applied Machine Learning in Health Care course.\nThis quiz is a 20-minute-long closed-book assessment component that carries a weightage of 40% for Understanding of Concep...,There is no information in the handouts about how many quizzes are scheduled for the Applied Machine Learning in Health Care course. The retrieved sections describe a quiz as a 20‑minute closed‑bo...
IoT evaluation criteria,"The evaluation criteria for IoT Networks, Architectures and Applications is based on the marks/grades obtained in the mid-term and end-term evaluations along with other components as defined in th...","The evaluation criteria are a Mid‑Term Exam (30 %), Lab Assignments (20 %), a Coursera Certification (10 %) and an End‑Term project (demonstration, viva and report) worth 40 %. This breakdown is d..."
IoT topics covered,"The topics covered in IoT Networks, Architectures and Applications include basics of networking, OSI and TCP/IP models, addressing, and TCP connections; what is IoT and why is IoT; applications of...","The course covers basics of networking, OSI and TCP/IP models, addressing and TCP connections; an introduction to what IoT is and why it matters; IoT applications together with their potential and..."
Theory of Computation credits,"The number of credits for Theory of Computation is 3. This information is supported by multiple contexts, including the course title and credits section in Context 1 and the repeated mention of 3 ...","The Theory of Computation course carries three credits. This information is listed in the credits section of the Course Handout – ToC (Jan‑June).pdf, where the course code CSE2721 is shown with a ..."
Theory of Computation evaluation criteria,**Evaluation Criteria for Theory of Computation:**\n\nThe evaluation criteria for Theory of Computation is based on the marks/grades obtained in the Mid-semester exam along with other assessments ...,**Evaluation criteria for Theory of Computation (CSE2721)** \n\n| Component | Weightage | Timing / Remarks | CO Mapping |\n|-----------|-----------|------------------|------------|\n| Quiz 1 & 2 ...


,question_label,model,sources,answer
0,Generative AI evaluation criteria,meta-llama/Llama-3.1-8B-Instruct,Generative AI and LLMs_CourseHandout.pdf,**Evaluation Criteria for Generative AI and LLMs:**\n\nThe evaluation criteria for Generative AI and LLMs is based on the relative grading method outlined in the university's academic regulations....
1,Generative AI evaluation criteria,openai/gpt-oss-120b,Generative AI and LLMs_CourseHandout.pdf,"**Evaluation criteria**\n\n- **Quiz** – 20 % of the final grade (20 MCQs, 1 mark each, 20‑30 min). \n- **Assignments 1 & 2** – each 20 % (total 40 %); programming assignments evaluated offline wi..."
2,Healthcare evaluation criteria,meta-llama/Llama-3.1-8B-Instruct,2025_course_handout_AMCH.pdf,"The evaluation criteria for Applied Machine Learning in Health Care involves a relative grading method, where the final grade will be based on the marks obtained in various assessment components, ..."
3,Healthcare evaluation criteria,openai/gpt-oss-120b,2025_course_handout_AMCH.pdf,"The course is assessed through four components – two quizzes worth a combined 20 % (each quiz 10 %), an assignment worth 20 %, a mid‑term presentation worth 20 % and a final project worth 40 %, wi..."
4,How many quiz are there in machine learning for healthcare?,meta-llama/Llama-3.1-8B-Instruct,2025_course_handout_AMCH.pdf,There is one quiz in the Applied Machine Learning in Health Care course.\nThis quiz is a 20-minute-long closed-book assessment component that carries a weightage of 40% for Understanding of Concep...
5,How many quiz are there in machine learning for healthcare?,openai/gpt-oss-120b,2025_course_handout_AMCH.pdf,There is no information in the handouts about how many quizzes are scheduled for the Applied Machine Learning in Health Care course. The retrieved sections describe a quiz as a 20‑minute closed‑bo...
6,IoT evaluation criteria,meta-llama/Llama-3.1-8B-Instruct,"IoT Networks, Architectures and Applications-CSE 2023 Batch.pdf","The evaluation criteria for IoT Networks, Architectures and Applications is based on the marks/grades obtained in the mid-term and end-term evaluations along with other components as defined in th..."
7,IoT evaluation criteria,openai/gpt-oss-120b,"IoT Networks, Architectures and Applications-CSE 2023 Batch.pdf","The evaluation criteria are a Mid‑Term Exam (30 %), Lab Assignments (20 %), a Coursera Certification (10 %) and an End‑Term project (demonstration, viva and report) worth 40 %. This breakdown is d..."
8,IoT topics covered,meta-llama/Llama-3.1-8B-Instruct,"IoT Networks, Architectures and Applications-CSE 2023 Batch.pdf","The topics covered in IoT Networks, Architectures and Applications include basics of networking, OSI and TCP/IP models, addressing, and TCP connections; what is IoT and why is IoT; applications of..."
9,IoT topics covered,openai/gpt-oss-120b,"IoT Networks, Architectures and Applications-CSE 2023 Batch.pdf","The course covers basics of networking, OSI and TCP/IP models, addressing and TCP connections; an introduction to what IoT is and why it matters; IoT applications together with their potential and..."


In [14]:
summary_dir = ROOT / 'results_summary'
summary_dir.mkdir(exist_ok=True)

answer_matrix.to_csv(summary_dir / 'retrieval_answer_matrix.csv')
source_matrix.to_csv(summary_dir / 'retrieval_source_matrix.csv')
matrix_df.to_csv(summary_dir / 'retrieval_details.csv', index=False)

if not compare_df.empty:
    compare_df.to_csv(summary_dir / 'generator_comparison_details.csv', index=False)

print(f'Saved summary files to: {summary_dir.resolve()}')


Saved summary files to: /Users/Lenovo/Desktop/sem 6/Gen_AI group assignment/assignment 1/results_summary
